In [8]:
import formulallm.formula as f

import io
import contextlib

from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent, FunctionCallingAgentWorker
from llama_index.llms.ollama import Ollama
from llama_index.core import PromptTemplate

In [9]:
llama3 = Ollama(model="llama3", base_url='http://localhost:11434', temperature=0.0, request_timeout=600)

In [11]:
react_agent = ReActAgent.from_tools(llm=llama3, verbose=True)

In [12]:
react_system_header_str = """\

You are designed to help with a variety of tasks, from answering questions \
to providing summaries to other types of analyses.

## Tools
You should rely solely on your natural language processing capabilities to complete the tasks. You should not use any external tools.

## Output Format
To answer the question, please use the following format.
```
Thought: Provide your reasoning or analysis here.
Answer: Provide your final answer here.
```

Please ensure that your responses are clear and concise, following the structure provided above.

## Current Conversation
Below is the current conversation consisting of interleaving human and assistant messages.
"""


In [13]:
react_system_prompt = PromptTemplate(react_system_header_str)

In [14]:
react_agent.update_prompts({"agent_worker:system_prompt": react_system_prompt})

In [16]:
prompt = """
This is the Formula DSL domain-partial model pair:
domain Mapping
{
  Component ::= new (id: Integer, utilization: Real).
  Processor ::= new (id: Integer).
  Mapping   ::= new (c: Component, p: Processor).

  // The utilization must be > 50
  invalidUtilization :- c is Component, c.utilization <= 50.

  badMapping :- p is Processor, 
                s = sum(0.0, { c.utilization |
                               c is Component, Mapping(c, p) }), s > 100.

  conforms no badMapping, no invalidUtilization.
}

partial model pm of Mapping
{
  c1 is Component(0, x).
  c2 is Component(1, y).
  p1 is Processor(0).
  p2 is Processor(1).
  Mapping(c1, p1).
  Mapping(c2, p1).
}

The partial model is unsolvable. Here is the explanation of why it is unsolvable:
The model is unsolvable because there are two conflicts that cannot be satisfied simultaneously. 
The first conflict is due to the utilization of components being less than or equal to 50.

The second conflict is related to the mapping of components to processors. 
The badMapping condition states that the total utilization of components mapped to a processor exceeds 100. Since there are two components (c1 and c2) mapped to processor p1, the total utilization would be x + y, which could exceed 100 if x or y is greater than 50.

These two conflicts cannot be satisfied simultaneously, making the model unsolvable.

Your task is to come up with a fix of the domain constraint to make the model solvable.
"""

In [17]:
response = react_agent.chat(prompt)

print(str(response))

> Running step 47bd03c3-c6b6-4640-919c-e4565190bf5b. Step input: 
This is the Formula DSL domain-partial model pair:
domain Mapping
{
  Component ::= new (id: Integer, utilization: Real).
  Processor ::= new (id: Integer).
  Mapping   ::= new (c: Component, p: Processor).

  // The utilization must be > 50
  invalidUtilization :- c is Component, c.utilization <= 50.

  badMapping :- p is Processor, 
                s = sum(0.0, { c.utilization |
                               c is Component, Mapping(c, p) }), s > 100.

  conforms no badMapping, no invalidUtilization.
}

partial model pm of Mapping
{
  c1 is Component(0, x).
  c2 is Component(1, y).
  p1 is Processor(0).
  p2 is Processor(1).
  Mapping(c1, p1).
  Mapping(c2, p1).
}

The partial model is unsolvable. Here is the explanation of why it is unsolvable:
The model is unsolvable because there are two conflicts that cannot be satisfied simultaneously. 
The first conflict is due to the utilization of components being less than or 